In [3]:
import numpy as np
from scipy.special import jv, jn_zeros
from scipy.optimize import brentq
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

# ======================================================================
# 1. Parámetros físicos y conversión adimensional
# ======================================================================
# Parámetros físicos base (ejemplo)
d = 15e-9          # Profundidad NV [m]
D1_phys = 2e-12    # Difusividad región 1 [m²/s]
D2_phys = 2e-10    # Difusividad región 2 [m²/s]
kappa_phys = 150e-9  # Coef. absorción [m/s]
L_phys = 1500e-9    # Altura total [m]
R_phys = 1500e-9    # Radio cilindro [m]
z_c_phys = 50e-9    # Posición interfaz [m]

# Conversión a adimensionales
D_ref = D2_phys
T_D = d**2/D_ref

D1 = D1_phys/D_ref
D2 = 1.0           # D2/D_ref = 1 por definición
kappa = (kappa_phys*d)/D_ref
L = L_phys/d
R = R_phys/d
z_c = z_c_phys/d

# ======================================================================
# 2. Solución radial (modos de Bessel)
# ======================================================================
def radial_equation(alpha):
    return alpha*jv(1, alpha*R) - (kappa/((z_c + (L-z_c))/(z_c/D1 + (L-z_c)/D2)))*jv(0, alpha*R)

alpha_roots = []
alpha_current = 1e-3
n_modes_radial = 25

while len(alpha_roots) < n_modes_radial:
    try:
        root = brentq(radial_equation, alpha_current, alpha_current + 0.1)
        alpha_roots.append(root)
        alpha_current = root + 0.1
    except:
        alpha_current += 0.1
alpha_n = np.array(alpha_roots)

# ======================================================================
# 3. Solución axial (paralelizado)
# ======================================================================
def solve_axial(alpha):
    def axial_eq(Lambda):
        beta1 = np.sqrt((Lambda**2 - alpha**2*D1)/D1)
        beta2 = np.sqrt((Lambda**2 - alpha**2*D2)/D2)
        return (D1*beta1*np.cos(beta2*(L - z_c)) - kappa*D2*beta2/(D1*beta1)*np.sin(beta2*(L - z_c)))*np.sin(beta1*z_c) + \
               (kappa*beta1*np.cos(beta2*(L - z_c)) + D2*beta2*np.sin(beta2*(L - z_c)))*np.cos(beta1*z_c)
    
    roots = []
    Lambda_current = alpha*np.sqrt(D2) + 1e-3
    m_max = 25
    
    while len(roots) < m_max:
        try:
            root = brentq(axial_eq, Lambda_current, Lambda_current + 0.1)
            roots.append(root)
            Lambda_current = root + 0.1
        except:
            Lambda_current += 0.1
    return roots

axial_roots = Parallel(n_jobs=4)(delayed(solve_axial)(alpha) for alpha in alpha_n)

# ======================================================================
# 4. Cálculo de G(t)
# ======================================================================
# Configuración espacial
N_r = 1000
N_z = 1000
rho_vals = np.linspace(0, R, N_r)
z_vals = np.linspace(0, L, N_z)

# Precomputar modos radiales
R_modes = jv(0, alpha_n[:, None]*rho_vals)
norms = 0.5*R**2*(jv(0, alpha_n*R)**2 + jv(1, alpha_n*R)**2)
R_modes /= np.sqrt(norms[:, None])

# Configuración temporal
t_phys = np.logspace(-12, -3, 50)
t_dim = t_phys/T_D

# Cálculo vectorizado de G(t)
G = np.zeros_like(t_dim)
for i, t in enumerate(t_dim):
    decay = np.exp(-np.abs(np.array(axial_roots)) * t)
    integral = 0
    
    for n in range(n_modes_radial):
        for m in range(len(axial_roots[n])):
            # Cálculo del modo axial
            Lambda = axial_roots[n][m]
            beta1 = np.sqrt((Lambda**2 - alpha_n[n]**2*D1)/D1)
            beta2 = np.sqrt((Lambda**2 - alpha_n[n]**2*D2)/D2)
            
            Z = np.piecewise(z_vals, 
                            [z_vals <= z_c, z_vals > z_c],
                            [lambda z: np.cos(beta1*z) - (kappa/(D1*beta1))*np.sin(beta1*z),
                             lambda z: (np.cos(beta1*z_c) - kappa/(D1*beta1)*np.sin(beta1*z_c)) * \
                                      (np.tan(beta2*(L - z_c))*np.sin(beta2*(z - z_c)) + np.cos(beta2*(z - z_c)))])
            
            # Normalización
            Z /= np.sqrt(np.trapz(Z**2, z_vals))
            
            # Integración
            R_part = R_modes[n]
            integrand = (3*z_vals**2 - rho_vals**2)/(rho_vals**2 + z_vals**2)**2.5
            integrand *= np.outer(R_part, Z)
            
            integral += decay[n, m] * np.trapz(np.trapz(integrand, z_vals), rho_vals)
    
    G[i] = (5/(16*np.pi)) * integral

# ======================================================================
# 5. Graficación
# ======================================================================
plt.figure(figsize=(10,6))
plt.loglog(t_phys, G/np.max(G), 'o-', color='navy')
plt.xlabel('Tiempo físico (s)')
plt.ylabel('G(t)/G(0)')
plt.grid(True, which='both', linestyle='--', alpha=0.7)
plt.title('Función de Correlación - Versión Simplificada')
plt.show()

C:\Users\AllanGonzalez\AppData\Local\Temp\ipykernel_18284\3873587281.py:116: RuntimeWarning: invalid value encountered in divide
  integrand = (3*z_vals**2 - rho_vals**2)/(rho_vals**2 + z_vals**2)**2.5


ValueError: non-broadcastable output operand with shape (1000,) doesn't match the broadcast shape (1000,1000)